# 7. Explore annotated MSI datasets in METASPACE

This notebook searches public METASPACE metadata without downloading imzML/ibd files. Start broadly to learn the values present in the database, inspect biological and acquisition metadata, open individual dataset pages, and only then refine the filters. The exported JSON is consumed by the existing `query --filters` command.

In [ ]:
from pathlib import Path

from IPython.display import display

from msi_autoencoder_wrapper.dataset_management.exploration import DatasetExplorer

explorer = DatasetExplorer(source="metaspace")

## Inspect supported filters

These fields are passed directly to the official METASPACE `SMInstance.datasets(...)` method. `exclude_dataset_ids` is the only local filter; it is applied after discovery and is not sent to METASPACE.

In [ ]:
explorer.available_filters()

## Start with a broad query

METASPACE contains many public mouse datasets. A broad first query avoids assuming spelling or metadata values that are not actually present. The result table includes organism, organ, polarity, processing status, image dimensions, annotation databases, and a direct dataset URL.

In [ ]:
broad_filters = {
    "organism": "Mouse",
    "exclude_dataset_ids": [],
}
results = explorer.search(broad_filters)
display(results.head(20))
print(f"Found {len(results)} datasets")

Summarize values present in the returned records before choosing stricter filters. Empty values mean that the submitter did not provide that metadata.

In [ ]:
for column in ["organism_parts", "polarity", "processing_status", "databases"]:
    print(f"\n{column}")
    display(results[column].value_counts(dropna=False).head(20))

## Refine the provider query

Use values observed above. This example searches names containing `liver` among public mouse datasets; this combination currently returns public records. Change or remove `nameMask`; optionally add `polarity`, `ionisation_source`, `analyzer_type`, `maldi_matrix`, `group_id`, or `project_id`. A term such as `bladder` may legitimately return no records if METASPACE currently has no matching public dataset.

In [ ]:
filters = {
    "organism": "Mouse",
    "nameMask": "liver",
    "exclude_dataset_ids": [],
}
results = explorer.search(filters)
display(results)

`rejected()` is normally empty for METASPACE because this source returns the provider's datasets directly. Unlike PRIDE, it does not reconstruct image/annotation pairs from project files or reject unsupported sidecar formats.

In [ ]:
explorer.rejected()

## Inspect one complete source record

The table is intentionally compact. Retrieve a selected record to inspect all metadata supplied by METASPACE before accepting it.

In [ ]:
if not results.empty:
    dataset_id = results.iloc[0]["dataset_id"]
    dataset_record = explorer.source.get_dataset_metadata(dataset_id)
    display(dataset_record)
    print(results.iloc[0]["project_url"])

## Exclude reviewed datasets and export

Open the URLs and list unsuitable IDs below. Manual exclusions are stored in the exported configuration and applied by both the notebook explorer and the CLI query.

In [ ]:
excluded_dataset_ids = []
if excluded_dataset_ids:
    explorer.exclude(excluded_dataset_ids)
display(explorer.results())

In [ ]:
output_path = Path("assets/configs/datasets/metaspace_mouse_liver.json")
explorer.export_config(output_path)

## Use the exported configuration

Run `manage_datasets.py query --source metaspace --filters assets/configs/datasets/metaspace_mouse_liver.json --selection workspace/datasets/selections/metaspace_mouse_liver.json`. Review the selection, then pass it to `download --source metaspace`. Querying and metadata review do not download MSI binary data.